In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../')
import mlflow
import torch
import numpy as np
import os
from src.utils import helpers
from src.utils import visualization as vis
from src.utils.model_registry import MLFlowRegistry
from src.core.params import BaseParams
from src.core.model_wrapper import ModelWrapper
from src.data.datamodule import CrowdDataModule
from src.utils import inspection
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
inspection.inspect_model('convnext_base')

In [ ]:
model, params = MLFlowRegistry().load_model('models:/crowd_counting/75')

In [ ]:
params

In [ ]:
# import numpy as np
# dummy_input = np.random.randint(0, 256, size=(2, 704, 1056, 3), dtype=np.uint8)
# model.model.eval()
model.predict(torch.randn(1, 3, 704, 1056))

In [ ]:
datamodule = CrowdDataModule(params=params)
datamodule.setup(stage=None)

In [ ]:
model.evaluate(datamodule.val_dataloader())

In [ ]:
datamodule.val_dataloader().batch_size

In [ ]:
"""{'mae': 105.06348419189453,
 'mbe': -6.24075174331665,
 'nae': 0.8911219239234924,
 'rmse': 340.9920959472656,
 'mask_dice': 0.491184800863266,
 'mask_iou': 0.3255434036254883}"""

In [ ]:
sample, target, count = helpers.get_sample_from_dm(datamodule, index=11)
sample.shape

In [ ]:
from torchvision.transforms import v2
from PIL import Image


from src.data import transform_sample
original_image = Image.open('/teamspace/studios/this_studio/workspace/crowd_counting/testing_images/input/0580.jpg').convert('RGB')
image = transform_sample.preprocess(original_image, params)
# image = np.array([original_image])

In [ ]:
result  = model.predict(image)
result = (float(result[0][0]), result[1])
result

In [ ]:
result = (result[0]['count'], result[0]['density_map'])

In [ ]:
vis.visualize_sample(original_image, pred=result, cmap='jet')

In [ ]:
from torchlens.options import VisualizationOptions
vis.visualize_model_graph(model.model, sample=torch.randn(1, 3, 256, 256), options=VisualizationOptions(view='rolled', direction='leftright', depth=4, file_format='png'))